# 01. Exploratory Data Analysis (EDA) - SaaS Subscription Metrics
**Author:** Ravikant Yadav  
**Date:** June 23, 2026  

This notebook walks through an in-depth exploratory data analysis of our SaaS subscription database. We will load and profile the raw cleaned datasets to verify data volumes, identify trends, inspect KPIs, and evaluate user demographic distributions.


In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set styles for beautiful visual storytelling
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (11, 6)
plt.rcParams["font.size"] = 11

PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data" / "cleaned"

# Verify files exist
for csv_file in DATA_DIR.glob("*.csv"):
    print(f"Dataset found: {csv_file.name} ({csv_file.stat().st_size / 1024:.1f} KB)")


## 1. Load Primary Tables
Let's load the primary tables: `users.csv`, `subscriptions.csv`, `payments.csv`, and `plans.csv`.


In [ ]:
users = pd.read_csv(DATA_DIR / "users.csv")
subscriptions = pd.read_csv(DATA_DIR / "subscriptions.csv")
payments = pd.read_csv(DATA_DIR / "payments.csv")
plans = pd.read_csv(DATA_DIR / "plans.csv")

print(f"Users shape: {users.shape}")
print(f"Subscriptions shape: {subscriptions.shape}")
print(f"Payments shape: {payments.shape}")
print(f"Plans shape: {plans.shape}")


## 2. Demographic Analysis of User Base
Let's check the size of the companies using our SaaS platform, as well as their regions and how they heard about us (acquisition channels).


In [ ]:
# Distribution of acquisition channels
channel_counts = users['acquisition_channel'].value_counts()
print("--- Acquisition Channels ---")
print(channel_counts)

# Distribution of region
region_counts = users['region'].value_counts()
print("\n--- Geographic Regions ---")
print(region_counts)

# Distribution of company size
company_counts = users['company_size'].value_counts()
print("\n--- Company Size Segments ---")
print(company_counts)


## 3. Subscription & Pricing Profiling
Let's look at the active plans, monthly pricing distributions, and total Monthly Recurring Revenue (MRR) by plan tier.


In [ ]:
# Merge subscriptions with plans to get subscription details
sub_plan = pd.merge(subscriptions, plans, on="plan_id", how="inner")
print(sub_plan.groupby("plan_name")['monthly_price_x'].agg(['count', 'mean', 'sum']))


## 4. Recurring Revenue Trends (MRR/ARR)
Let's load the pre-computed monthly KPIs and visualize the Monthly Recurring Revenue (MRR) and Annual Recurring Revenue (ARR) timeline to evaluate the business's overall health.


In [ ]:
monthly_kpis = pd.read_csv(DATA_DIR / "monthly_kpis.csv")
monthly_kpis['month'] = pd.to_datetime(monthly_kpis['month'])

print("--- Pre-computed Monthly KPIs Sample ---")
print(monthly_kpis.head())

# Plot MRR trend
plt.figure(figsize=(12, 5))
plt.plot(monthly_kpis['month'], monthly_kpis['mrr'], marker='o', color='#2b5c8f', linewidth=2.5)
plt.title("Monthly Recurring Revenue (MRR) Timeline (2022 - 2025)", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Month", fontsize=12)
plt.ylabel("MRR ($)", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()


## 5. Summary Findings
1. Our user base is geographically diverse, with strong representation in several international markets.
2. Enterprise and Business tiers contribute the lion's share of recurring revenue, even though Basic/Starter plans have high logo volumes.
3. The MRR trend shows clear stability but flat-to-linear growth over the last 12 months, highlighting the absolute necessity of diving into churn dynamics.
